In this notebook we process the data from the MD-Simulations

In [1]:
import pandas as pd

# We drop all duplicate molecules and keep the best (=lowest) score
def clean_dataset(df):
    n_duplicates = df["smiles"].duplicated().sum()
    print(f"Dataset contains {n_duplicates} duplicates.")
    
    assert not df["target"].isna().any(), "Dataset contains molecules without a score."
    
    clean_df = df.sort_values(by="target").drop_duplicates(["smiles"], keep="first")
    # Avoid any information leakage by having the data ordered
    clean_df = clean_df.sample(frac=1.0, random_state=0).reset_index(drop=True)
    return clean_df

We start by cleaning the Enamine datasets that have been used in the literature to evaluate active learning.

In [ ]:
for ds in ["unprocessed_Enamine10k_scores.csv", "unprocessed_Enamine50k_scores.csv"]:
    df = pd.read_csv(ds)
    clean_df = clean_dataset(df)
    
    clean_df.to_csv(ds.removeprefix("unprocessed_"), index=False)

Dataset contains 3 duplicates.
Dataset contains 7 duplicates.


The results from docking and MMGBSA/MMPBSA have been prepared in the `results.csv` file. We now extract the necessary columns for bayesian optimization.

In [3]:
results_df = pd.read_csv("results.csv")
results_df = results_df.drop(columns=["orig_smiles"])
results_df = results_df.rename(columns={"prot_smiles": "smiles"})
results_df.head()

,name,smiles,mmgbsa_score,mmgbsa_score_sem,mmpbsa_score,mmpbsa_score_sem,vina_score
0,ZINCtL000014Qw97,COc1ccc([N+](=O)[O-])cc1COC(=O)c1[nH]c2c(Br)cc...,-54.684265,0.720370,-43.393463,0.880179,-8.206
1,ZINCsU00001b6zrh,O=C(OCc1cccc2c1CCCC2)c1[nH]c2c(Br)cccc2c1CCCO,-53.876397,0.630838,-44.423265,0.706064,-9.915
2,ZINCsU00001aYbfK,O=C(OCc1c(Cl)oc2ccccc12)c1[nH]c2c(Br)cccc2c1CCCO,-53.526174,0.241943,-43.616633,0.286676,-9.199
3,ZINCsQ00001e1YP6,COc1ncc(Br)c(C)c1COC(=O)c1[nH]c2c(Br)cccc2c1CCCO,-52.599412,0.895439,-43.444357,0.975881,-8.151
4,ZINCtV000002EVzt,C[C@@H]1CC[C@H](OC(=O)CCc2c(-c3ccccc3)[nH]c3c(...,-52.518792,0.441199,-40.200800,0.351907,-9.343


In [7]:
results_df[["name", "smiles", "mmgbsa_score"]].rename(columns={"mmgbsa_score":"target"}).to_csv("MCL1-mmgbsa.csv")
results_df[["name", "smiles", "vina_score"]].rename(columns={"vina_score":"target"}).to_csv("MCL1-vina.csv")